# Exploring hotel data with pandas

Imagine helping a visitor find an affordable, well-rated hotel. **Can we find a quote below EUR 100 per night with a guest rating of at least 4 out of 5?** Before comparing prices, we need to check what each price buys.

A data scientist starts with questions, not a list of commands. We will ask:

1. **What information do we have, and what does one row represent?**
2. **Are all quotes for the same number of nights? How can we compare them?**
3. **What does a typical night cost, and how much do nightly prices vary?**
4. **Does every quote have a rating from both review sources?**
5. **Which quotes meet our nightly budget and guest-rating requirements?**
6. **Which qualifying quotes are cheapest per night?**

Each question leads to a small piece of code. Read the output, then use it to answer the question. We will load, inspect, transform, summarise, filter, and sort a DataFrame.

This notebook starts from one CSV hosted on the course GitHub repository. You do **not** need to complete `01_pandas_basics.ipynb` first or join datasets in class.

## Before we start: a table in Python

**pandas** is a Python library for working with tables. A **DataFrame** is a table with labelled rows and columns. Here, each row is a price quote for a hotel stay; its columns describe the quote and the hotel. We will check whether a hotel can appear in more than one row.

Use the course Python environment and run the cells from top to bottom. Keep an internet connection: the loading cell reads `hotel_vienna_quotes.csv` directly from the course's GitHub repository. You do not need a local copy of the CSV, a manual download, or a Git pull to obtain the data.

`read_csv` can read a CSV from a web address as well as from a local file path. It loads the data into Python's memory; it does not save a CSV on your computer.

In [ ]:
# pd is the short name we use to access pandas.
import pandas as pd

## 1. What information do we have, and what does one row represent?

We use hotel price quotes from the [Hotels Europe teaching data](https://gabors-data-analysis.com/datasets/hotels-europe/). These are historical teaching data, not current offers. A small file has been prepared for this lesson; it is not a complete or representative list of hotels.

Load the file and inspect it. Which columns describe the stay, and which describe the hotel? Look for location and period information as well as prices and ratings.

Reading the CSV does not change the file hosted on GitHub.

In [ ]:
# Read the supplied CSV directly from GitHub; an internet connection is needed.
url = "https://raw.githubusercontent.com/ulrichwohak/Coding-1-Introduction-to-Python/main/lectures/lecture04-pandas-basics/hotel_vienna_quotes.csv"
hotels = pd.read_csv(url)

# Inspect the first rows, including the location and period columns.
hotels.head()

### Keep the columns relevant to our questions

We will work with the columns below. These are **definitions**, not findings about which values occur in our sample; we will investigate those values in code.

| Column | Meaning |
| --- | --- |
| `hotel_id` | Hotel identifier, not a quantity to average |
| `neighbourhood` | Area label recorded in the data |
| `stars` | Hotel star category; different from a guest rating |
| `price` | Total quoted price in EUR for the stay |
| `nnights` | Number of nights covered by that quote |
| `ratings` | Average guest rating on a 0–5 scale |
| `rating_count` | Number of reviews behind that guest rating |
| `distance` | Distance from the main city centre, in miles |
| `ratingta` | Alternative guest rating from Tripadvisor |

`hotels["price"]` selects one column as a **Series**. `hotels[["price", "nnights"]]` selects a **list of columns** and returns a DataFrame. The inner brackets make the list; the outer brackets select from the table.

In [ ]:
# Select our working columns; copy() gives us a separate table to work on.
columns = [
    "hotel_id", "neighbourhood", "stars", "price", "nnights",
    "ratings", "rating_count", "distance", "ratingta",
]
hotels = hotels[columns].copy()

# Preview the first five rows, not the entire dataset.
hotels.head()

Read across one row: which value is the total price, and how many nights does it cover?

The numbers at the far left are the **index**, or row labels. They are not the `hotel_id` column. Later, filtered tables will retain their original row labels.

`head()` is a **method**: an operation we ask the DataFrame to perform. Parentheses call the method; `head(3)` would request three rows instead of five.

In [ ]:
# shape gives (number of rows, number of columns); no parentheses are needed.
hotels.shape

In [ ]:
# Inspect column names, non-missing counts, and the types pandas read.
hotels.info()

Use `shape` and `info()` to answer: how many rows and selected columns are there? Which columns contain numbers, and which contain text?

In `info()`, `int64` and `float64` are numeric types; `object` here contains text. A numeric type does not make every column a useful measurement: `hotel_id` is still just an identifier.

Does the number of rows also tell us the number of different hotels? Count distinct identifiers before deciding.

In [ ]:
# Count distinct hotel IDs, rather than rows of quotes.
hotels["hotel_id"].nunique()

Compare this result with the number of rows from `shape`. What does a difference tell us about repeated hotel IDs?

Now investigate the star categories. Which values appear? Which is most common among these quotes? Are any star values missing?

In [ ]:
# Count each observed star category, including any missing values.
hotels["stars"].value_counts(dropna=False)

## 2. Are all quotes for the same number of nights?

Before sorting by price, inspect `nnights`. A total stay price alone does not tell us the cost of one night.

Which stay lengths occur? Does the output include any missing values or zero nights? A nightly-price calculation needs an observed, positive denominator.

In [ ]:
# Discover the stay lengths; dropna=False also reveals missing values, if any.
hotels["nnights"].value_counts(dropna=False)

### Create the price per night

Divide each quote's total price by its number of nights. pandas performs the calculation row by row across the two columns, without a loop.

Keep the original `price` column and store the result in a new column called `price_per_night`.

In [ ]:
# Convert each total stay price into an average price per night.
hotels["price_per_night"] = hotels["price"] / hotels["nnights"]

# Compare the original inputs with the new column for the first few quotes.
hotels[["hotel_id", "price", "nnights", "price_per_night"]].head(10)

Pick a quote covering more than one night and check the division yourself. When do `price` and `price_per_night` coincide, and when do they differ?

Does a longer stay automatically have a lower nightly price? Read the displayed quotes rather than assuming that it does. This is an average per night for the quoted stay, not a price offered for booking one night separately.

## 3. What does a typical night cost, and how much do nightly prices vary?

Is EUR 100 per night a generous budget or a restrictive one in this sample? `describe()` gives a quick numerical overview. Select meaningful measurements rather than averaging identifiers or calendar fields.

In [ ]:
# Summarise the nightly price we created, guest ratings, and distance.
hotels[["price_per_night", "ratings", "distance"]].describe()

Read the **price_per_night** column first:

- `count`: how many non-missing nightly prices contribute to the summary.
- `mean`: the average nightly price.
- `min` and `max`: the lowest and highest nightly prices.
- `50%`: the median, or middle nightly price when the nightly prices are ordered.

What is the median nightly price? How does it compare with the mean? What range do you see, and what does that tell you about relying on a single average?

`25%` and `75%` are other percentile cutoffs; `std` describes spread. Today, focus on finding and reading the counts, typical values, and range, not on calculating statistics.

**Discuss:** Our rule will be *below* EUR 100 per night. Should a quote costing exactly EUR 100 per night qualify?

## 4. Does every quote have a rating from both review sources?

A blank rating does not mean a rating of zero. Before comparing reviews, check whether information is missing.

In [ ]:
# isna() marks missing entries True and observed entries False.
missing_ratings = hotels[["ratings", "ratingta"]].isna()

# sum() counts the True entries separately for each column.
missing_ratings.sum()

Which rating column has missing values, and how many? Is the missingness the same for both sources?

Do not replace missing ratings with zero or discard rows automatically. Think about which information the question needs. If a primary rating is missing, we cannot confirm that the quote meets our rating requirement; a missing alternative rating need not prevent us from using an observed primary rating.

This also explains why `describe()` reports a separate non-missing `count` for each column.

## 5. Which quotes meet our nightly budget and guest-rating requirements?

Start with one condition: a **nightly** price below EUR 100. Comparing a column with a number checks every row, so we do not need to write a loop. Using the original total `price` here would answer a different question.

In [ ]:
# Each row gets True if its nightly price is strictly below 100.
within_budget = hotels["price_per_night"] < 100
within_budget.head()

Use that sequence of True/False values to choose rows. Read `.loc[rows, columns]` as: **which rows, then which columns?**

With only `.loc[within_budget]`, we select the matching rows and keep every column.

In [ ]:
# Keep the rows marked True; hotels itself is not changed.
affordable = hotels.loc[within_budget]
affordable.head()

Now require a guest rating of at least 4 as well. For pandas columns, `&` means both conditions must hold. Put each comparison in parentheses; do not use Python's `and` between two Series.

In [ ]:
# A quote must pass both tests, row by row.
good_value = (hotels["price_per_night"] < 100) & (hotels["ratings"] >= 4)

# Keep the total, stay length, and nightly price visible together.
shortlist = hotels.loc[
    good_value,
    ["hotel_id", "price", "nnights", "price_per_night", "ratings", "rating_count", "distance"],
]
shortlist.shape

How many quotes meet both criteria? Read the row count from `shortlist.shape`. That is not necessarily the number of distinct hotels.

`shortlist` is a new result; we have not removed the other rows from `hotels` or changed the CSV. We chose the name `good_value` for our two criteria. It is not proof that these offers are bargains: stay length, rating, location, and review count may point in different directions.

## 6. Which qualifying quotes are cheapest per night?

A filter selects rows; a sort puts those rows in a useful order. Show the five lowest **nightly** prices among the qualifying quotes. Keep `price` and `nnights` visible so we can still see the total commitment.

In [ ]:
# sort_values() returns a sorted table; assigning it keeps that result.
shortlist = shortlist.sort_values("price_per_night")

# The default order is ascending, so the lowest nightly prices appear first.
shortlist.head(5)

Read the first row: what is the cheapest qualifying nightly price? How many nights does that quote cover, and what is the total price?

**Discuss:** Would you automatically choose it? Compare stay lengths, distances, ratings, and review counts. The source provides period indicators, not exact check-in dates; do not assume these are interchangeable offers for one specific trip.

Notice that the row labels did not restart at zero. Sorting changed the order of rows, not the hotels' identifiers.

## Main lesson complete

We went from a question to an inspected table, created a comparable unit of price, and built a reproducible shortlist. Someone else can rerun our cells and see the calculation and selection criteria.

**Try a variation:** A different visitor cares most about location. Find quotes below EUR 120 **per night** with a guest rating of at least 4, but show the five **closest to the centre**, rather than the cheapest. Which part of the filter changes? Which part of the sorting changes?

The remaining sections are optional extensions. The main lesson does not depend on them.

## Optional A. How far away are these hotels in kilometres?

The source reports miles. We can create a new column by applying the same calculation to a whole column. One mile is 1.609344 kilometres.

In [ ]:
# Make an independent copy before adding information to the selected table.
shortlist = shortlist.copy()

# Multiply every distance by the conversion factor; no loop is needed.
shortlist["distance_km"] = shortlist["distance"] * 1.609344
shortlist[["hotel_id", "price_per_night", "distance_km"]].head()

## Optional B. Do higher-star hotels have higher typical nightly prices here?

Use all quotes, not just our shortlist. `groupby("stars")` separates rows by star category; selecting `price_per_night` and calling `median()` summarises nightly prices within each category.

Each quote contributes one observation. Hotels with more quotes therefore contribute more rows; this is a summary of quotes, not an equally weighted summary of hotels.

In [ ]:
# Calculate one median nightly price per star category, using all quotes.
median_price_by_stars = hotels.groupby("stars")["price_per_night"].median()
median_price_by_stars

Which star category has the highest median nightly price? Does the median always rise with the star category?

Describe what the output shows for these observations. It does not show that changing a hotel's star category would cause its price to change.

## Optional C. How can we share the shortlist as a file?

A DataFrame in memory is not yet a saved file. We can export a new CSV without overwriting our input. The `scratch` folder below is created inside the current working directory and is ignored by Git in this course repository.

In [ ]:
# Path helps us create a folder and locate the file we save.
from pathlib import Path

# Create the output folder if it does not already exist.
Path("scratch").mkdir(exist_ok=True)

# Save the shortlist, without adding the pandas row labels as a CSV column.
shortlist.to_csv("scratch/lecture04_hotel_shortlist.csv", index=False)

# Display the full path so we know where the new file was saved.
Path("scratch/lecture04_hotel_shortlist.csv").resolve()

## Finding help and sources

You do not need to memorise every DataFrame method. These official pages let you check what a method accepts and returns:

- [Load a CSV: `read_csv`](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.read_csv.html)
- [Read numerical summaries: `DataFrame.describe`](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.describe.html)
- [Select rows and columns: `DataFrame.loc`](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.loc.html)
- [Choose the sort order: `DataFrame.sort_values`](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.sort_values.html)

The supplied CSV combines stay prices and hotel characteristics without calculating nightly prices for you. Its reproducible preparation is documented in [data/README.md](../../data/README.md). For the original data definitions and educational-use terms, see the [Hotels Europe dataset description](https://gabors-data-analysis.com/datasets/hotels-europe/). For attribution of the adapted course materials, see [NOTICE.md](../../NOTICE.md).